In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFrameTransformationsDemo") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created")

In [ ]:
employees = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/employees.csv")

departments = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/departments.csv")

In [ ]:
employees.show()
departments.show()

In [ ]:
it_employees = employees.filter(
    employees.department_id == 10
)
it_employees.show()

In [ ]:
high_salary = employees.filter(
    employees.salary > 100000
)

high_salary.show()

In [ ]:
filtered_df = employees.filter(
    (employees.salary > 80000) &
    (employees.city == "Vadodara")
)

filtered_df.show()

In [ ]:
selected_df = employees.select(
    "name",
    "job_title",
    "salary"
)

selected_df.show()

In [ ]:
employees_with_annual_salary = employees.withColumn(
    "annual_salary",
    employees.salary * 12
)

employees_with_annual_salary.show()

In [ ]:
dropped_df = employees.drop("age")

dropped_df.show()

In [ ]:
dropped_df = employees.drop(
    "age",
    "city"
)

dropped_df.show()

In [ ]:
from pyspark.sql.functions import avg

department_salary = employees.groupBy(
    "department_id"
).agg(
    avg("salary").alias("average_salary")
)

department_salary.show()

In [ ]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    min,
    max
)

department_stats = employees.groupBy(
    "department_id"
).agg(
    count("*").alias("employee_count"),
    sum("salary").alias("total_salary"),
    avg("salary").alias("average_salary"),
    min("salary").alias("minimum_salary"),
    max("salary").alias("maximum_salary")
)

department_stats.show()

In [ ]:
employee_details = employees.join(
    departments,
    employees.department_id == departments.department_id,
    "inner"
)

employee_details.show()

In [ ]:
employee_details = employees.join(
    departments,
    employees.department_id == departments.department_id,
    "inner" # inner, left, right, outer, left_semi, left_anti
).select(
    employees.employee_id,
    employees.name,
    employees.job_title,
    employees.salary,
    departments.department_name,
    departments.location
)

employee_details.show()

In [ ]:
cities = employees.select(
    "city"
)

cities.show()

In [ ]:
unique_cities = employees.select(
    "city"
).distinct()

unique_cities.show()

In [ ]:
employees.show(10, truncate = False)

In [ ]:
employee_count = employees.count()

print(f"Employee Count: {employee_count}")

In [ ]:
from pyspark.sql.functions import col
high_salary_count = employees.filter(
    col("salary") > 100000
).count()

print("High Salary Employees:", high_salary_count)

In [ ]:
rows = employees.take(5)

for row in rows:
    # print(row)
    print(
        row.employee_id,
        row.name,
        row.salary
    )

In [ ]:
rows = employees.collect()

for row in rows:
    print(row)

In [ ]:
from pyspark.sql.functions import (
    col,
    lit,
    when,
    concat
)

In [ ]:
df = employees.filter(
    col("salary") > 100000
)

df.show()

In [ ]:
df = employees.withColumn(
    "company",
    lit("Zensar Technologies")
)

df.show()

In [ ]:
df = employees.withColumn(
    "country",
    lit("India")
)
df.show()

In [ ]:
df = employees.withColumn(
    "salary_level",
    when(col("salary") >= 150000, "Very High")
    .when(col("salary") >= 100000, "High")
    .when(col("salary") >= 70000, "Medium")
    .otherwise("Low")
)

df.select(
    "name",
    "salary",
    "salary_level"
).show()

In [ ]:
df = employees.withColumn(
    "employee_description",
    concat(
        col("name"),
        lit(" - "),
        col("job_title")
    )
)

df.select(
    "employee_id",
    "employee_description"
).show(truncate=False)

In [ ]:
df = employees.withColumn(
    "employee_label",
    concat(
        col("name"),
        lit(" | "),
        col("job_title"),
        lit(" | Salary: "),
        col("salary").cast("string")
    )
)

df.select(
    "employee_id",
    "employee_label"
).show(truncate=False)